## Gold Layer - Medication Availability by Region

### Description
This notebook creates a Gold-level aggregated dataset showing medication availability per U.S. region.

The dataset aggregates pharmacy inventory to provide a region-level view of medication stock and pharmacy coverage.

### Source Tables
capstone.silver.inventory  
capstone.silver.medications  
capstone.silver.pharmacy  

### Target Table
capstone.gold.medication_availability_region_gold

### Transformations
- Join inventory with pharmacy to obtain region  
- Join medications to get medication name  
- Filter pharmacies that have stock available  
- Aggregate by region and medication  
- Calculate total available quantity  
- Calculate pharmacy coverage per medication

## 1. Setup Environment


In [0]:
# Setup environment
# Use capstone catalog and ensure Gold schema exists

spark.sql("USE CATALOG capstone")
spark.sql("CREATE SCHEMA IF NOT EXISTS gold")

## 2. Create Gold Table

In [0]:
%sql
CREATE OR REPLACE TABLE capstone.gold.medication_availability_region_gold AS

SELECT
    region,
    medication_id,
    medication_name,

    COUNT(DISTINCT pharmacy_id) AS pharmacy_count,
    SUM(quantity) AS total_quantity

FROM capstone.gold.disease_medication_pharmacy_gold

WHERE quantity > 0

GROUP BY
    region,
    medication_id,
    medication_name;

In [0]:
%sql
-- Preview the Gold dataset

SELECT *
FROM capstone.gold.medication_availability_region_gold
ORDER BY region, medication_name

## 3. Document Table **Columns**

In [0]:
%sql
-- Document Gold Table Columns


ALTER TABLE capstone.gold.medication_availability_region_gold
ALTER COLUMN region COMMENT 'U.S. region where the pharmacy is located';

ALTER TABLE capstone.gold.medication_availability_region_gold
ALTER COLUMN medication_id COMMENT 'Unique identifier of the medication';

ALTER TABLE capstone.gold.medication_availability_region_gold
ALTER COLUMN medication_name COMMENT 'Standardized medication name';

ALTER TABLE capstone.gold.medication_availability_region_gold
ALTER COLUMN pharmacy_count COMMENT 'Number of pharmacies in the region with medication available';

ALTER TABLE capstone.gold.medication_availability_region_gold
ALTER COLUMN total_quantity COMMENT 'Total quantity of medication available in the region';

## 4. Gold Data Quality Checks

In [0]:
%sql

-- Total number of rows in the Gold dataset
SELECT 
'Total Rows' AS check_type,
COUNT(*) AS result
FROM capstone.gold.medication_availability_region_gold

UNION ALL

-- Check for missing region values
SELECT
'Missing Region',
COUNT(*)
FROM capstone.gold.medication_availability_region_gold
WHERE region IS NULL

UNION ALL

-- Check for missing medication identifiers
SELECT
'Missing Medication ID',
COUNT(*)
FROM capstone.gold.medication_availability_region_gold
WHERE medication_id IS NULL

UNION ALL

-- Check for invalid negative quantities
SELECT
'Negative Quantity',
COUNT(*)
FROM capstone.gold.medication_availability_region_gold
WHERE total_quantity < 0

UNION ALL

-- Check for duplicate business keys
SELECT
'Duplicate Region + Medication',
COUNT(*)
FROM (
    SELECT region, medication_id
    FROM capstone.gold.medication_availability_region_gold
    GROUP BY region, medication_id
    HAVING COUNT(*) > 1
)


## 5. Validate Gold Table

In [0]:
%sql

SELECT *
FROM capstone.gold.medication_availability_region_gold
LIMIT 20;

## 6. KPI - Top medications by region

In [0]:
%sql
CREATE OR REPLACE VIEW capstone.kpi.kpi_top_medications_by_region AS

SELECT
    region,
    medication_name,
    SUM(total_quantity) AS total_available_quantity
FROM capstone.gold.medication_availability_region_gold
GROUP BY
    region,
    medication_name
ORDER BY
    region,
    total_available_quantity DESC;

## 7. KPI - Regions with Lowest Medication Availability

In [0]:
%sql
CREATE OR REPLACE VIEW capstone.kpi.kpi_regions_lowest_availability AS

SELECT
    region,
    SUM(total_quantity) AS total_medication_stock
FROM capstone.gold.medication_availability_region_gold
GROUP BY
    region
ORDER BY
    total_medication_stock ASC;